# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** The `mlcroissant` library allows us to list available record sets and fields via the metadata interface. Throughout, we always refer to Croissant schema elements (such as record sets, fields, columns) by their `@id`.

In [ ]:
# List all record sets (@id and name)
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = getattr(metadata, 'recordSet', [])

print("Available Record Sets:")
record_set_ids = []
for rs in record_sets:
    # Use explicit attribute access in case of dataclasses
    rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
    rs_name = getattr(rs, 'name', None)
    print(f"  - @id: {rs_id}\tname: {rs_name}")
    record_set_ids.append(rs_id)

if record_sets:
    example_rs = record_sets[0]
    example_rs_id = getattr(example_rs, '@id', getattr(example_rs, 'id', None))
    print(f"\nExample record set: {example_rs_id}")

    # Print fields in the example record set
    print("\nFields (and @id) in the first record set:")
    for field in getattr(example_rs, 'fields', getattr(example_rs, 'field', [])):
        f_id = getattr(field, '@id', getattr(field, 'id', None))
        f_name = getattr(field, 'name', None)
        print(f"  - @id: {f_id}\tname: {f_name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the record set and field `@id`s from the overview.

**Note:** We will extract each available record set, referencing them by their `@id`. DataFrames are keyed using these `@id`s.

In [ ]:
# Extract data from all record sets by @id
dataframes = {}

for record_set in record_sets:
    rs_id = getattr(record_set, '@id', getattr(record_set, 'id', None))
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded Record Set: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
        print("\n---\n")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}\n")

# For demonstration, choose the first loaded record set for EDA
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break

if main_rs_id is not None:
    print(f"Using record set for further analysis: {main_rs_id}")
    print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No non-empty record set found for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

All field references use their Croissant `@id`.

In [ ]:
# Choose numeric field for analysis: search for float/integer columns
df = dataframes[main_rs_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", numeric_fields)

# For illustration, pick the first numeric field (or fallback to 'Age' if present)
if 'Age' in df.columns:
    numeric_field_id = 'Age'
elif numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    print('No numeric field found; exploratory analysis will be limited.')
    numeric_field_id = None

# Filter records based on a threshold (e.g., Age > 50)
threshold = 50
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if available (e.g., 'Sex' or first non-numeric column)
    group_fields = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < len(df)/2]
    if 'Sex' in group_fields:
        group_field = 'Sex'
    elif group_fields:
        group_field = group_fields[0]
    else:
        group_field = None

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
else:
    filtered_df = df
    print('No numeric field to analyze or filter.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

All axes and legends reference variables by their Croissant column `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not filtered_df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found or filtered data empty for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the Croissant schema and the `mlcroissant` library to load, inspect, process, and visualize the FAIR² clinical dataset. 
- **Schema-driven:** All extractions and references use `@id`s for accuracy and reproducibility.
- **Extensible:** You can further analyze, join, or model on the loaded DataFrames using the same principles shown here.

*For more advanced analyses, see [mlcroissant documentation](https://mlcommons.github.io/croissant/).*